# GenerativeAURORA — Step 4: Build the Diffusion Model

## Purpose

Define the complete **Conditional Denoising Diffusion Probabilistic Model (DDPM)** used by GenerativeAURORA to generate full-day GHI trajectories.

The model is conditioned on three sources of information:

1. **Weather context (`cond_x`)** — a 16-hour × 10-feature historical context window.
2. **Sky regime (`regime`)** — Clear, Cloudy, or Overcast.
3. **Site identity (`site`)** — one of the five study locations.

The model contains four main components:

- a **1D CNN + GRU condition encoder** for the weather context,
- learned **regime and site embeddings**,
- a **1D U-Net** for denoising the GHI trajectory,
- a **linear diffusion noise schedule** with 200 diffusion steps.

## Workflow position

```text
download_nsrdb
      ↓
merge_nsrdb
      ↓
eda_nsrdb
      ↓
step3_build_sequences
      ↓
step4_build_model  ← THIS NOTEBOOK
      ↓
step5_train
      ↓
step6_evaluate
      ↓
step7_conformal
```

## Input / output

**Conceptual input:** tensors produced by Step 3, including the GHI target sequence, weather context, sky-regime label, and site label.

**Output of this step:** a PyTorch model definition. This notebook does **not** train the model or save model weights; those tasks belong to Step 5.

> **Implementation note:** The architecture below preserves the logic of the supplied `step4_build_model.py`. The notebook mainly reorganizes the code into explanatory, executable sections for Colab.

## 1. Import required packages

The model is implemented in **PyTorch**. This step only imports the libraries needed to define and test the architecture.

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch version:", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

PyTorch version: 2.11.0+cu130
CUDA available  : True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Define model hyperparameters

| Parameter | Value | Meaning |
|---|---:|---|
| `HORIZON` | 16 | Length of the GHI trajectory to generate |
| `LOOKBACK` | 16 | Length of the weather conditioning window |
| `N_FEATURES` | 10 | Number of conditioning variables from Step 3 |
| `N_REGIMES` | 3 | Clear / Cloudy / Overcast |
| `N_SITES` | 5 | Phoenix / Los Angeles / Denver / Miami / Seattle |
| `T_DIFFUSION` | 200 | Number of diffusion timesteps |
| `EMBED_DIM` | 64 | Shared embedding dimension |
| `HIDDEN_DIM` | 128 | Base U-Net channel width |

These dimensions should remain consistent with the sequence format created in Step 3.

In [ ]:
HORIZON     = 16
LOOKBACK    = 16
N_FEATURES  = 10
N_REGIMES   = 3
N_SITES     = 5
T_DIFFUSION = 200
EMBED_DIM   = 64
HIDDEN_DIM  = 128

print("HORIZON     :", HORIZON)
print("LOOKBACK    :", LOOKBACK)
print("N_FEATURES  :", N_FEATURES)
print("N_REGIMES   :", N_REGIMES)
print("N_SITES     :", N_SITES)
print("T_DIFFUSION :", T_DIFFUSION)
print("EMBED_DIM   :", EMBED_DIM)
print("HIDDEN_DIM  :", HIDDEN_DIM)

HORIZON     : 16
LOOKBACK    : 16
N_FEATURES  : 10
N_REGIMES   : 3
N_SITES     : 5
T_DIFFUSION : 200
EMBED_DIM   : 64
HIDDEN_DIM  : 128


## 3. Construct the diffusion noise schedule

DDPM training gradually adds Gaussian noise to a clean target trajectory.

The original implementation uses a **linear beta schedule**:

\[
\beta_t \in [10^{-4}, 0.02]
\]

and then computes:

\[
\alpha_t = 1 - \beta_t
\]

\[
\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s
\]

`alpha_bar` is used to transform the clean GHI trajectory directly into a noisy trajectory at an arbitrary diffusion timestep.

In [ ]:
def make_beta_schedule(T, beta_start=1e-4, beta_end=0.02):
    """
    Create the linear DDPM beta schedule used by GenerativeAURORA.

    Parameters
    ----------
    T : int
        Number of diffusion timesteps.
    beta_start : float
        Initial noise variance.
    beta_end : float
        Final noise variance.

    Returns
    -------
    tuple of torch.Tensor
        betas, alphas, and cumulative alpha products (alpha_bar).
    """
    betas = torch.linspace(beta_start, beta_end, T)
    alphas = 1.0 - betas
    alpha_bar = torch.cumprod(alphas, dim=0)
    return betas, alphas, alpha_bar


betas, alphas, alpha_bar = make_beta_schedule(T_DIFFUSION)

print("Betas shape     :", tuple(betas.shape))
print("Alphas shape    :", tuple(alphas.shape))
print("Alpha-bar shape :", tuple(alpha_bar.shape))
print("First beta      :", float(betas[0]))
print("Last beta       :", float(betas[-1]))

Betas shape     : (200,)
Alphas shape    : (200,)
Alpha-bar shape : (200,)
First beta      : 9.999999747378752e-05
Last beta       : 0.019999999552965164


## 4. Encode the diffusion timestep

The model must know **which noise level** is being applied to the sample.

`TimestepEmbedding` converts the integer diffusion timestep into a continuous vector using sinusoidal features, followed by a small multilayer projection. The resulting embedding is injected into the residual blocks of the U-Net.

In [ ]:
class TimestepEmbedding(nn.Module):
    """
    Encode a diffusion timestep as a learned projection of sinusoidal features.

    Input
    -----
    t : torch.Tensor
        Integer diffusion timesteps with shape (B,).

    Output
    ------
    torch.Tensor
        Timestep embedding with shape (B, embed_dim).
    """

    def __init__(self, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim
        self.proj = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.SiLU(),
            nn.Linear(embed_dim * 2, embed_dim),
        )

    def sinusoidal(self, t):
        half = self.embed_dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(
                half, dtype=torch.float32, device=t.device
            ) / (half - 1)
        )
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        return emb

    def forward(self, t):
        emb = self.sinusoidal(t)
        return self.proj(emb)

## 5. Encode the weather context

Step 3 provides the conditioning window as:

\[
X \in \mathbb{R}^{B \times 16 \times 10}
\]

The condition encoder processes the weather context in two stages:

1. **1D convolutions** extract local temporal patterns from the ten conditioning variables.
2. A **2-layer GRU** summarizes the 16-step sequence into a fixed-size 64-dimensional vector.

The final hidden state of the GRU is normalized with `LayerNorm` and becomes the weather-conditioning embedding.

In [ ]:
class ConditionEncoder(nn.Module):
    """
    Encode the 16-hour x 10-feature weather context into one vector.

    Input
    -----
    x : torch.Tensor
        Shape (B, LOOKBACK, N_FEATURES).

    Output
    ------
    torch.Tensor
        Shape (B, EMBED_DIM).
    """

    def __init__(self, n_features=N_FEATURES, lookback=LOOKBACK, embed_dim=EMBED_DIM):
        super().__init__()
        self.conv1 = nn.Conv1d(n_features, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.gru = nn.GRU(64, embed_dim, batch_first=True, num_layers=2)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x.permute(0, 2, 1)       # (B, N_FEATURES, LOOKBACK)
        x = F.silu(self.conv1(x))
        x = F.silu(self.conv2(x))
        x = x.permute(0, 2, 1)       # (B, LOOKBACK, 64)
        _, h = self.gru(x)           # (2, B, EMBED_DIM)
        cond = self.norm(h[-1])       # (B, EMBED_DIM)
        return cond

## 6. Define the residual block used by the U-Net

`ResBlock1D` refines the U-Net feature maps while preserving information through a residual connection.

Each block receives two external signals:

- the **diffusion timestep embedding**, and
- the **combined conditioning embedding**.

Both embeddings are projected to the current channel dimension and added to the intermediate feature representation.

In [ ]:
class ResBlock1D(nn.Module):
    """
    Residual 1D convolutional block with timestep and condition injection.

    Parameters
    ----------
    channels : int
        Number of input/output channels.
    embed_dim : int
        Dimension of timestep and conditioning embeddings.
    """

    def __init__(self, channels, embed_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, channels)
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(8, channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.t_proj = nn.Linear(embed_dim, channels)
        self.c_proj = nn.Linear(embed_dim, channels)

    def forward(self, x, t_emb, c_emb):
        h = F.silu(self.norm1(x))
        h = self.conv1(h)
        h = h + self.t_proj(t_emb).unsqueeze(-1)
        h = h + self.c_proj(c_emb).unsqueeze(-1)
        h = F.silu(self.norm2(h))
        h = self.conv2(h)
        return x + h

## 7. Build the conditional 1D diffusion U-Net

The central denoising network operates on a noisy 16-hour GHI trajectory:

\[
x_t \in \mathbb{R}^{B \times 16}
\]

### Conditioning branch

The model combines four embeddings:

- timestep embedding,
- sky-regime embedding,
- site embedding,
- weather-context embedding.

These are concatenated and fused into one conditioning vector.

### U-Net branch

The U-Net contains:

- an input projection,
- two encoder levels,
- a bottleneck,
- two decoder levels,
- skip connections,
- an output projection that predicts one noise value for each horizon position.

The network therefore estimates:

\[
\epsilon_\theta(x_t, t, X, R, S)
\]

where `X` is weather context, `R` is sky regime, and `S` is site identity.

In [ ]:
class DiffusionUNet(nn.Module):
    """
    Conditional 1D U-Net used to predict diffusion noise for a GHI trajectory.

    Inputs
    ------
    x_noisy : (B, HORIZON)
        Noisy GHI trajectory at diffusion timestep t.
    t : (B,)
        Integer diffusion timesteps.
    cond_x : (B, LOOKBACK, N_FEATURES)
        Historical weather-conditioning window.
    regime : (B,)
        Sky-regime indices.
    site : (B,)
        Site indices.

    Output
    ------
    torch.Tensor
        Predicted noise with shape (B, HORIZON).
    """

    def __init__(
        self,
        horizon=HORIZON,
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        n_regimes=N_REGIMES,
        n_sites=N_SITES,
    ):
        super().__init__()

        self.horizon = horizon
        self.embed_dim = embed_dim

        # Conditioning modules
        self.t_emb = TimestepEmbedding(embed_dim)
        self.regime_emb = nn.Embedding(n_regimes, embed_dim)
        self.site_emb = nn.Embedding(n_sites, embed_dim)
        self.cond_enc = ConditionEncoder(embed_dim=embed_dim)

        self.cond_fuse = nn.Sequential(
            nn.Linear(embed_dim * 4, embed_dim * 2),
            nn.SiLU(),
            nn.Linear(embed_dim * 2, embed_dim),
        )

        # U-Net encoder
        self.input_proj = nn.Conv1d(1, hidden_dim, kernel_size=1)
        self.enc1 = ResBlock1D(hidden_dim, embed_dim)
        self.down1 = nn.Conv1d(
            hidden_dim, hidden_dim * 2, kernel_size=2, stride=2
        )

        self.enc2 = ResBlock1D(hidden_dim * 2, embed_dim)
        self.down2 = nn.Conv1d(
            hidden_dim * 2, hidden_dim * 4, kernel_size=2, stride=2
        )

        # Bottleneck
        self.bottleneck = ResBlock1D(hidden_dim * 4, embed_dim)

        # U-Net decoder
        self.up1 = nn.ConvTranspose1d(
            hidden_dim * 4, hidden_dim * 2, kernel_size=2, stride=2
        )
        self.dec1 = ResBlock1D(hidden_dim * 4, embed_dim)

        self.up2 = nn.ConvTranspose1d(
            hidden_dim * 4, hidden_dim, kernel_size=2, stride=2
        )
        self.dec2 = ResBlock1D(hidden_dim * 2, embed_dim)

        # Output projection
        self.out_norm = nn.GroupNorm(8, hidden_dim * 2)
        self.out_proj = nn.Conv1d(hidden_dim * 2, 1, kernel_size=1)

    def forward(self, x_noisy, t, cond_x, regime, site):
        # Build conditioning vector
        t_e = self.t_emb(t)
        r_e = self.regime_emb(regime)
        s_e = self.site_emb(site)
        c_e = self.cond_enc(cond_x)
        cond = self.cond_fuse(torch.cat([t_e, r_e, s_e, c_e], dim=-1))

        # Reshape noisy trajectory for Conv1D
        h = x_noisy.unsqueeze(1)
        h = self.input_proj(h)

        # Encoder
        e1 = self.enc1(h, t_e, cond)
        h = self.down1(e1)
        e2 = self.enc2(h, t_e, cond)
        h = self.down2(e2)

        # Bottleneck
        h = self.bottleneck(h, t_e, cond)

        # Decoder + skip connections
        h = self.up1(h)
        h = torch.cat([h, e2], dim=1)
        h = self.dec1(h, t_e, cond)

        h = self.up2(h)
        h = torch.cat([h, e1], dim=1)
        h = self.dec2(h, t_e, cond)

        # Predict noise epsilon
        h = F.silu(self.out_norm(h))
        h = self.out_proj(h)
        return h.squeeze(1)

## 8. Wrap the U-Net in the full GenerativeAURORA DDPM

`GenerativeAURORA` adds the diffusion process around the U-Net.

### Forward diffusion (`q_sample`)

A clean trajectory \(x_0\) is transformed into a noisy trajectory \(x_t\):

\[
x_t = \sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\epsilon
\]

where \(\epsilon \sim \mathcal{N}(0,I)\).

### Training pass (`forward`)

For each batch, the model:

1. samples a random diffusion timestep,
2. adds noise to the clean GHI trajectory,
3. asks the U-Net to predict that noise,
4. computes mean-squared error between true and predicted noise.

### Sampling (`sample`)

At inference time, the model starts from Gaussian noise and repeatedly applies the reverse diffusion step from timestep `T-1` down to `0`, producing multiple GHI scenarios for the same conditioning information.

In [ ]:
class GenerativeAURORA(nn.Module):
    """
    Full conditional DDPM wrapper for GenerativeAURORA.

    The wrapper contains the diffusion schedule and the conditional U-Net.
    It supports both training (`forward`) and generation (`sample`).
    """

    def __init__(self, T=T_DIFFUSION):
        super().__init__()
        self.T = T
        self.unet = DiffusionUNet()

        betas, alphas, alpha_bar = make_beta_schedule(T)
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alpha_bar", alpha_bar)

    def q_sample(self, x0, t, noise=None):
        """
        Add Gaussian noise to clean trajectories using the DDPM forward process.
        """
        if noise is None:
            noise = torch.randn_like(x0)

        ab = self.alpha_bar[t].unsqueeze(1)
        x_t = ab.sqrt() * x0 + (1 - ab).sqrt() * noise
        return x_t, noise

    def forward(self, x0, cond_x, regime, site):
        """
        Perform one DDPM training step and return the noise-prediction loss.
        """
        B = x0.shape[0]
        t = torch.randint(0, self.T, (B,), device=x0.device)
        x_noisy, noise = self.q_sample(x0, t)
        noise_pred = self.unet(x_noisy, t, cond_x, regime, site)
        loss = F.mse_loss(noise_pred, noise)
        return loss

    @torch.no_grad()
    def sample(self, cond_x, regime, site, n_scenarios=50):
        """
        Generate multiple GHI scenarios by reversing the diffusion process.

        Parameters
        ----------
        cond_x : torch.Tensor
            Weather context, shape (B, LOOKBACK, N_FEATURES).
        regime : torch.Tensor
            Sky-regime indices, shape (B,).
        site : torch.Tensor
            Site indices, shape (B,).
        n_scenarios : int
            Number of generated scenarios per conditioning sample.

        Returns
        -------
        torch.Tensor
            Generated scenarios with shape (B * n_scenarios, HORIZON).
        """
        B = cond_x.shape[0]
        device = cond_x.device

        cond_x = cond_x.repeat_interleave(n_scenarios, dim=0)
        regime = regime.repeat_interleave(n_scenarios)
        site = site.repeat_interleave(n_scenarios)

        # Start from pure Gaussian noise
        x = torch.randn(B * n_scenarios, self.unet.horizon, device=device)

        # Reverse diffusion loop
        for step in reversed(range(self.T)):
            t_batch = torch.full(
                (B * n_scenarios,),
                step,
                dtype=torch.long,
                device=device,
            )
            noise_pred = self.unet(x, t_batch, cond_x, regime, site)

            beta = self.betas[step]
            alpha = self.alphas[step]
            ab = self.alpha_bar[step]

            # DDPM reverse step, matching the supplied implementation
            x = (1.0 / alpha.sqrt()) * (
                x - (beta / (1 - ab).sqrt()) * noise_pred
            )

            if step > 0:
                x = x + beta.sqrt() * torch.randn_like(x)

        return x

## 9. Instantiate the model and inspect its size

At this stage we only construct the architecture. No training data are required.

The parameter count provides a quick check of model complexity and helps document the architecture before training.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GenerativeAURORA(T=T_DIFFUSION).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Device              :", device)
print("Total parameters    :", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

Device              : cuda
Total parameters    : 5,095,329
Trainable parameters: 5,095,329


## 10. Verify the training forward pass

Before moving to Step 5, confirm that the architecture accepts the dimensions expected from Step 3.

For this smoke test:

- `x0` has shape `(B, 16)`
- `cond_x` has shape `(B, 16, 10)`
- `regime` has shape `(B,)`
- `site` has shape `(B,)`

The training forward pass should return a **single scalar loss**.

In [ ]:
B = 4

x0 = torch.randn(B, HORIZON, device=device)
cond_x = torch.randn(B, LOOKBACK, N_FEATURES, device=device)
regime = torch.randint(0, N_REGIMES, (B,), device=device)
site = torch.randint(0, N_SITES, (B,), device=device)

loss = model(x0, cond_x, regime, site)

print("x0 shape       :", tuple(x0.shape))
print("cond_x shape   :", tuple(cond_x.shape))
print("regime shape   :", tuple(regime.shape))
print("site shape     :", tuple(site.shape))
print("Loss value     :", round(float(loss.item()), 6))
print("Loss dimensions:", tuple(loss.shape), "(scalar expected)")

assert loss.ndim == 0, "Training loss should be a scalar."
print("\nForward-pass shape check: PASSED")

x0 shape       : (4, 16)
cond_x shape   : (4, 16, 10)
regime shape   : (4,)
site shape     : (4,)
Loss value     : 1.014211
Loss dimensions: () (scalar expected)

Forward-pass shape check: PASSED


## 11. Verify scenario generation

The `sample()` method begins with random noise and performs the reverse diffusion process.

The original script uses 10 scenarios per sample for its direct architecture test. To keep a Colab smoke test lightweight, this notebook uses **2 scenarios per sample** here. This changes only the verification workload—not the model architecture or sampling algorithm.

With `B = 2` conditioning samples and `2` scenarios each, the expected output is:

\[
(2 \times 2,\;16) = (4,16)
\]

In [ ]:
cond_x_test = torch.randn(2, LOOKBACK, N_FEATURES, device=device)
regime_test = torch.randint(0, N_REGIMES, (2,), device=device)
site_test = torch.randint(0, N_SITES, (2,), device=device)

scenarios = model.sample(
    cond_x_test,
    regime_test,
    site_test,
    n_scenarios=2,
)

expected_shape = (4, HORIZON)

print("Generated scenarios shape:", tuple(scenarios.shape))
print("Expected shape           :", expected_shape)

assert tuple(scenarios.shape) == expected_shape, (
    f"Unexpected sample shape: {tuple(scenarios.shape)}"
)

print("Scenario-generation shape check: PASSED")

Generated scenarios shape: (4, 16)
Expected shape           : (4, 16)
Scenario-generation shape check: PASSED


## 12. Final architecture checklist

The Step 4 architecture is ready for Step 5 when all of the following checks pass:

- the diffusion schedule contains 200 timesteps,
- the weather context is accepted as `(B, 16, 10)`,
- regime labels are accepted as integer indices `0–2`,
- site labels are accepted as integer indices `0–4`,
- the training pass returns a scalar MSE loss,
- the sampling pass returns `(B × n_scenarios, 16)` trajectories,
- no model training is performed in this notebook.

**Next step:** use this `GenerativeAURORA` model in `step5_train.py` / Step 5 to train the diffusion model on the sequence files produced by Step 3.